In [ ]:
import tensorflow as tf
import tensorflow_ranking as tfr
import datetime
import numpy as np
import pandas as pd

In [ ]:
# quick tensorboard example:
# https://www.tensorflow.org/tensorboard/get_started

mnist = tf.keras.datasets.mnist

(x_train, y_train),(x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

def create_model():
  return tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation='softmax')
  ])

model = create_model()
model.compile( 
                optimizer = 'adam',
                loss = 'sparse_categorical_crossentropy',
                metrics = ['accuracy']
)

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard( log_dir=log_dir, histogram_freq=1 )

model.fit(
            x = x_train, 
            y = y_train, 
            epochs = 5, 
            validation_data = (x_test, y_test), 
            callbacks = [tensorboard_callback]
)

# run this in command line to see the tensorboard:    
# tensorboard --logdir logs/fit

LearnToRank Learning to Rank (LTR) deals with learning to optimally order a list of examples, given some context. 

For instance, in search applications, examples are documents and context is the serach query.  

In recommender applications, examples are items and context is item features (e.g. text description)/user features/session features

### Work through https://github.com/tensorflow/ranking/blob/master/tensorflow_ranking/examples/tf_ranking_tfrecord.py
I'm changed indenting, and added a lot of exploration and commenting

In [1]:
from absl import flags               # absl.flags defines a distributed command line system
                                     # At Google, they use flag systems to set default values for arguments. It's similar to argparse. 
                                     # They use their own flag system instead of argparse or sys.argv
import tensorflow as tf
import tensorflow_ranking as tfr

specify default model hyperparameter values:

In [2]:
flags.DEFINE_enum(                                                                          # Registers a flag whose value can be any string from enum_values
    name = "data_format",                                                                   # the flag name
    default = "example_list_with_context",                                                  # the default value of the flag 
    enum_values = ["example_list_with_context", "example_in_example", "sequence_example"],  # possible values of the string
    help = "Data format defined in data.py."                                                # the help message
)
flags.DEFINE_string(                                                                        # Registers a flag whose value can be any string
    name = "train_path",
    default = None,
    help = "Input file path used for training."
)
flags.DEFINE_string(
    name = "eval_path",
    default = None,
    help = "Input file path used for eval."
)
flags.DEFINE_string(
    name = "vocab_path",
    default = None,
    help = "Vocabulary path for query and document tokens."
)
flags.DEFINE_string(
    name = "model_dir",
    default = None,
    help = "Output directory for models."
)
flags.DEFINE_integer(
    name = "batch_size",
    default = 32,
    help = "The batch size for train."
)
flags.DEFINE_integer(
    name = "num_train_steps",
    default = 15000,
    help = "Number of steps for train."
)
flags.DEFINE_float(
    name = "learning_rate",
    default = 0.05,
    help = "Learning rate for optimizer."
)
flags.DEFINE_float(
    name = "dropout_rate",
    default = 0.8,
    help = "The dropout rate before output layer."
)
flags.DEFINE_list(                                        # Registers a flag whose value is a comma-separated list of strings
    name = "hidden_layer_dims",
    default = ["64", "32", "16"],
    help = "Sizes for hidden layers."
)
flags.DEFINE_integer(
    name = "list_size",
    default = None,
    help = "List size used for training. Use None for dynamic list size."
)
flags.DEFINE_integer(
    name = "group_size",
    default = 1,
    help = "Group size used in score function."
)
flags.DEFINE_string(
    name = "loss",
    default = "approx_ndcg_loss",
    help = "The RankingLossKey for the loss function."
)
flags.DEFINE_string(
    name = "weights_feature_name", 
    default = "",
    help = "The name of the feature where unbiased learning-to-rank " "weights are stored."
)
flags.DEFINE_bool(
    name = "listwise_inference", 
    default = False,
    help = "If true, exports accept `data_format` while serving.")
flags.DEFINE_bool(
    name = "use_document_interaction",
    default = False,
    help = "If True, use Document Interaction Network to capture cross-document "
           "interactions as additional features for scoring."
)
flags.DEFINE_integer(
    name = "num_attention_layers",
    default = 1,
    help = "number of attention layers. See "
            "`tfr.keras.layers.DocumentInteractionAttention`."
)
flags.DEFINE_integer(
    name = "num_attention_heads",
    default = 1,
    help = "number of self attention heads. See "
            "`tfr.keras.layers.DocumentInteractionAttention`."
)
flags.DEFINE_integer(
    name = "head_size",
    default = 128,
    help = "Size of attention head. See "
            "`tfr.keras.layers.DocumentInteractionAttention`."
)

FLAGS = flags.FLAGS

FLAGS

In [7]:
_LABEL_FEATURE = "relevance"
_PADDING_LABEL = -1
_EMBEDDING_DIMENSION = 20
_MASK = "mask"

Note that the number of collisions arising from feature hashing can be calculated using the solution to the birthday problem:

$$\begin{array}{lcl} 
n &=& \text{size (dimension) of hashing space required to achieve } c \text{ collisions (with feature space of dimension } k \text{)} \\
k &=& \text{dimension (number of unique values) of feature} \\
c &=& \text{expected number of collisions} \\
n &=& \Bigg( 1 - \Big(1-\displaystyle\frac{c}{k}\Big)^{\displaystyle\frac{1}{k-1}} \Bigg)^{-1} \\
&\approx& \displaystyle\frac{k^2}{c} \hspace{30mm} \text{(in fact, a very good approximation)}\\
\end{array}$$

(from https://booking.ai/dont-be-tricked-by-the-hashing-trick-192a6aae3087)


In [8]:
def context_feature_columns():
    """Returns context feature names to column definitions."""
    if FLAGS.vocab_path:                                                # if a path to a vocab file is provided then do this
        sparse_column = tf.feature_column.categorical_column_with_vocabulary_file(
            # https://www.tensorflow.org/api_docs/python/tf/feature_column/categorical_column_with_vocabulary_file
            # Use this when your inputs are in string or integer format, and you have a vocabulary file that maps each value to an integer ID
            # By default, out-of-vocabulary values are ignored
            key = "query_tokens",                                    	# A unique string identifying the input feature. It is used as the column name and the dictionary key for feature parsing configs, feature Tensor objects, and feature columns.
            vocabulary_file = FLAGS.vocab_path                          
        )
    else:                                                               # if a path to a vocab file is NOT provided, then do this 
        sparse_column = tf.feature_column.categorical_column_with_hash_bucket(
            # https://www.tensorflow.org/api_docs/python/tf/feature_column/categorical_column_with_hash_bucket
            # Represents sparse feature where ids are set by hashing.
            # hashing maps each category to an integer in a set of fixed size, potentially resulting in collisions (mapping of different unrelated categories to the same integer)
            key = "query_tokens", 
            hash_bucket_size = 100
        )
    # embed the sparse column created above:
    query_embedding_column = tf.feature_column.embedding_column(
        # https://www.tensorflow.org/api_docs/python/tf/feature_column/embedding_column
        categorical_column = sparse_column,
        dimension = _EMBEDDING_DIMENSION        # we defined this globally (same dim for all embedded features)
  )
    
    return {"query_tokens": query_embedding_column}

In [ ]:
def example_feature_columns(use_weight_feature=True):
    """Returns the example feature columns."""
    if FLAGS.vocab_path:
        sparse_column = tf.feature_column.categorical_column_with_vocabulary_file(
        key="document_tokens", vocabulary_file=FLAGS.vocab_path)
  else:
    sparse_column = tf.feature_column.categorical_column_with_hash_bucket(
        key="document_tokens", hash_bucket_size=100)
  document_embedding_column = tf.feature_column.embedding_column(
      sparse_column, _EMBEDDING_DIMENSION)
  feature_columns = {"document_tokens": document_embedding_column}
  if use_weight_feature and FLAGS.weights_feature_name:
    feature_columns[FLAGS.weights_feature_name] = (
        tf.feature_column.numeric_column(FLAGS.weights_feature_name,
                                         default_value=1.))
  return feature_columns